# 03 · CSP + classifier

## Like you're five
1. **CSP** is a clever mixer. We have 29 microphones; CSP finds the best *mixes* where left-thinking sounds loud and right-thinking sounds quiet (or the other way). Each trial becomes just **6 numbers**.
2. A **classifier** looks at those 6 numbers and says LEFT or RIGHT. It learned by studying examples with the right answers ("supervised learning").

We compare three classifiers (everything else stays identical):
- **LDA**: best simple baseline
- **Logistic regression**: gives probabilities, great for "confidence"
- **Linear SVM**: draws a wide-margin line between the classes

Why not feed raw EEG straight in? Too many numbers (29 × 1000) for only 40 trials per patient.

In [ ]:
# --- Setup (works on your laptop AND on Google Colab) ---
import os, sys
if "google.colab" in sys.modules and not os.path.exists("src"):
    # Colab: get the project first, e.g.  !git clone <your-repo-url> && %cd stroke-rehab-bci/notebooks
    pass
sys.path.append("..")          # so we can "import src"
# !pip install -q moabb mne scikit-learn pandas matplotlib   # uncomment on Colab
import numpy as np, pandas as pd, matplotlib.pyplot as plt


In [ ]:
from src.preprocessing import load_liu2024, encode_labels
from src.models import get_models
from src.evaluation import grouped_kfold, add_chance_flag
X, y_text, meta = load_liu2024(subjects=list(range(1, 11)))   # 10 patients for a quick look
y, enc = encode_labels(y_text); groups = meta["subject"].to_numpy()

### ⚠️ Golden rule: never split trials randomly
If patient 12's trials are in both train and test, the model can recognise *the person* instead of *the thought*. So we hold out **whole patients**. Here we use 5 patient-level folds (each patient lives in only one fold).

In [ ]:
for name, model in get_models().items():
    summary, trials = grouped_kfold(model, X, y, groups, n_splits=5)
    print(f"{name:18s} mean balanced accuracy = {summary.balanced_accuracy.mean():.3f}")

### CSP patterns (best effort)
This shows *where on the head* the CSP mixes look. It needs channel positions, which MOABB can provide through MNE epochs. If this cell errors, skip it and read the MOABB docs for your version.

In [ ]:
from moabb.datasets import Liu2024
from moabb.paradigms import LeftRightImagery
from src.features import make_csp
paradigm = LeftRightImagery(fmin=8, fmax=30, tmin=0, tmax=4, resample=250)
epochs, labels, _ = paradigm.get_data(Liu2024(), subjects=[1], return_epochs=True)
csp = make_csp(); csp.fit(epochs.get_data(), enc.transform(labels))
csp.plot_patterns(epochs.info, ch_type="eeg", units="a.u.", size=1.5)
plt.savefig("../figures/csp_patterns.png", dpi=150)